In [ ]:
import os, re, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceExceptions

from app.utils import Helper

helper = Helper()

In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = uc.Chrome(version_main=145)
wait = WebDriverWait(driver, 20)

driver.get("https://vahan.parivahan.gov.in/vahan4dashboard/vahan/vahan/view/reportview.xhtml")

STATE_DROPDOWN = "j_idt41_label" #monday #"j_idt41_label" #sunday
GET_STATES ="j_idt42_items li" #monday "#j_idt41_items li" #sunday
REFRESH = "j_idt74"#monday #"j_idt71" #sunday

def open_state_dropdown():
    el = wait.until(EC.element_to_be_clickable((By.ID,STATE_DROPDOWN)))
    driver.execute_script("arguments[0].click();", el)

def get_states():
    return wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, GET_STATES)
    ))

def close_dropdown():
    driver.find_element(By.CLASS_NAME, "ui-grid-row").click()
    time.sleep(0.3)

def select_option(dropdown_id, items_id, value):
    # open dropdown
    el = wait.until(EC.element_to_be_clickable((By.ID, f"{dropdown_id}_label")))
    driver.execute_script("arguments[0].click();", el)

    # get options
    items = wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, f"#{items_id} li")
    ))

    for item in items:
        label = item.get_attribute("data-label")
        if label and value.lower() in label.lower():
            driver.execute_script("arguments[0].click();", item)
            time.sleep(1.5)
            close_dropdown()
            return

    raise Exception(f"{value} not found")

# --- TEST STATE CLICKING ---
open_state_dropdown()
states = get_states()
years = list(range(2026, 2025, -1))  # 2026 → 2005
print("Total states:", len(states))

for i in range(1, len(states)):  # skip "All"
    open_state_dropdown()
    states = get_states()

    state = states[i]
    name = state.get_attribute("data-label")

    if not name:
        continue

    print("Clicking:", name)

    driver.execute_script("arguments[0].click();", state)
    time.sleep(1.5)
    close_dropdown()
   

    for year in years:
        print(f"\n--- YEAR: {year} ---")
        select_option("selectedYearType", "selectedYearType_items", "Calendar")
        time.sleep(0.2)
        select_option("selectedYear", "selectedYear_items", str(year))
        time.sleep(0.2)

        refresh_btn = wait.until(EC.element_to_be_clickable((By.ID, REFRESH)))
        driver.execute_script("arguments[0].click();", refresh_btn)

        print("Refreshed for year:", year)

        time.sleep(2)

driver.quit()

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import undetected_chromedriver as uc

options = uc.ChromeOptions()
options.add_argument("--start-maximized")

driver = uc.Chrome(
    options=options,
    version_main=146  # match your Chrome version
)

wait = WebDriverWait(driver, 20)

# driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

driver.get("https://vahan.parivahan.gov.in/vahan4dashboard/vahan/vahan/view/reportview.xhtml")


def click_dropdown(label_id_part):
    el = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, f"[id*='{label_id_part}_label']")))
    driver.execute_script("arguments[0].click();", el)


def select_from_dropdown(panel_id_part, text):
    items = wait.until(EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, f"[id*='{panel_id_part}_items'] li")))
    for item in items:
        if text.lower() in item.text.lower():
            driver.execute_script("arguments[0].click();", item)
            return
    raise Exception(f"{text} not found")


# --- STEP 1: SELECT STATE (skip first) ---
click_dropdown("j_idt33")  # state dropdown

states = wait.until(EC.presence_of_all_elements_located(
    (By.CSS_SELECTOR, "[id*='j_idt33_items'] li")))

for i in range(1, len(states)):  # skip first option
    click_dropdown("j_idt33")
    states = driver.find_elements(By.CSS_SELECTOR, "[id*='j_idt33_items'] li")

    state = states[i]
    print("STATE:", state.text)
    driver.execute_script("arguments[0].click();", state)

    time.sleep(1)

    # --- STEP 2: RTO (keep first real option) ---
    # click_dropdown("selectedRto")

    # rtos = wait.until(EC.presence_of_all_elements_located(
    #     (By.CSS_SELECTOR, "[id*='selectedRto_items'] li")))

    # if len(rtos) > 1:
    #     driver.execute_script("arguments[0].click();", rtos[1])  # skip "All"
    
    # time.sleep(1)

    # # --- STEP 3: Y AXIS ---
    # click_dropdown("yaxisVar")
    # select_from_dropdown("yaxisVar", "Maker")

    # # --- STEP 4: X AXIS ---
    # click_dropdown("xaxisVar")
    # select_from_dropdown("xaxisVar", "Vehicle Category")

    # # --- STEP 5: YEAR TYPE (Calendar) ---
    # click_dropdown("yearType")
    # select_from_dropdown("yearType", "Calendar")

    # # --- STEP 6: YEAR ---
    # click_dropdown("selectedYear")
    # select_from_dropdown("selectedYear", "2026")

    # --- STEP 7: REFRESH ---
    # refresh_btn = wait.until(EC.element_to_be_clickable(
    #     (By.CSS_SELECTOR, "[id*='j_idt'][id$='71']")))
    # driver.execute_script("arguments[0].click();", refresh_btn)

    # print("Refreshed:", state.text)

    # time.sleep(2)

    # # --- STEP 8: MONTH TRAVERSAL ---
    # months = ["JAN", "FEB", "MAR"]

    # for month in months:
    #     click_dropdown("selectMonth")
    #     select_from_dropdown("selectMonth", month)

    #     print("   Month:", month)
    #     time.sleep(1)

driver.quit()

STATE: 
STATE: In Crore
STATE: Actual Value


In [ ]:
import os, re, time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException

from app.utils import Helper

helper = Helper()

download_dir = r"C:\Users\rando\Downloads" 
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 20)

driver.get("https://vahan.parivahan.gov.in/vahan4dashboard/vahan/vahan/view/reportview.xhtml")

# --- CSS selectors for dropdown items ---
state_items_css  = "#j_idt33_items li"
rto_items_css    = "#selectedRto_items li"
yaxis_items_css  = "#yaxisVar_items li"
xaxis_items_css  = "#xaxisVar_items li"
year_items_css   = "#selectedYear_items li"

# --- Helper for safe clicks ---
def safe_click(locator, retries=3, delay=1):
    for attempt in range(retries):
        try:
            el = wait.until(EC.element_to_be_clickable(locator))
            driver.execute_script("arguments[0].click();", el)
            return
        except StaleElementReferenceException:
            print("Stale element, retrying:", locator)
            time.sleep(delay)
    raise Exception(f"Could not click {locator} after {retries} retries")

# --- Select Month ---
def select_month(month_label, retries=3):
    for attempt in range(retries):
        try:
            # Always re-open the dropdown to get a fresh DOM
            safe_click((By.ID, "groupingTable:selectMonth_label"))
            time.sleep(0.4)

            # Re-locate the overlay items each time
            month_items = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "#groupingTable\\:selectMonth_items li"))
            )

            # Find the desired month by text
            for m in month_items:
                if m.text.strip() == month_label:
                    driver.execute_script("arguments[0].click();", m)
                    return
        except StaleElementReferenceException:
            print(f"Stale element while selecting {month_label}, retrying...")
            time.sleep(0.5)
    raise Exception(f"Could not select month {month_label} after {retries} retries")

all_dataframes = []


safe_click((By.ID, "j_idt33_label")) #"j_idt41_label"
time.sleep(0.5)
states = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, state_items_css)))


for i in range(len(states)):
    safe_click((By.ID, "j_idt33_label"))
    time.sleep(0.6)
    states = driver.find_elements(By.CSS_SELECTOR, state_items_css)

    state = states[i]
    print("\nSTATE:", state.text)
    driver.execute_script("arguments[0].click();", state)
    time.sleep(1)

    # --- Wait for RTOs ---
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, rto_items_css)) > 1)
    safe_click((By.ID, "selectedRto_label"))
    time.sleep(1)
    rtos = driver.find_elements(By.CSS_SELECTOR, rto_items_css)

    for r in rtos[:1]:   # skip "All Offices"
        print("RTO:", r.text)
        driver.execute_script("arguments[0].click();", r)
        time.sleep(.5)

        # --- Set Y-Axis ---
        safe_click((By.ID, "yaxisVar_label"))
        time.sleep(0.5)
        for opt in driver.find_elements(By.CSS_SELECTOR, yaxis_items_css):
            if "Maker" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Set X-Axis = Month Wise ---
        safe_click((By.ID, "xaxisVar_label"))   # open dropdown
        time.sleep(0.5)

        xaxis_opts = driver.find_elements(By.CSS_SELECTOR, "#xaxisVar_items li")
        for opt in xaxis_opts:
            if "Vehicle Category" in opt.text or "Vehicle Category" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Set Year ---
        safe_click((By.ID, "selectedYear_label"))
        time.sleep(1)
        for opt in driver.find_elements(By.CSS_SELECTOR, year_items_css):
            if "2026" in opt.text:
                driver.execute_script("arguments[0].click();", opt)
                break

        # --- Click Refresh ---
        safe_click((By.ID, "j_idt71"))
        time.sleep(.5)
        print("\tRefreshed for", state.text, "->", r.text)
        
        
        # --- Select Month ---

        months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]
        for month in months:
            select_month(month)
            time.sleep(0.9)
            # --- Click Excel download ---
            safe_click((By.ID, "groupingTable:xls"))
            time.sleep(.1)

driver.quit()